# circuit-tracer native edge test: does the graph's prediction hold on the real model?

`circuit-tracer` builds an attribution graph on a transcoder **replacement** model and reports
which feature (and error) nodes influence a target logit. A graph edge is a candidate, not a
verdict. `saturn_pub.interop.circuit_tracer.native_edge_test` re-tests each edge on the **real**
Gemma-2-2B weights and asks two honest questions:

1. **native-necessity** — is removing this one feature individually load-bearing on the real
   model? (`native_necessary` / `native_effect_absent` / `inconclusive`). A single feature that
   is not individually necessary is *expected* under a redundant circuit and is **not** by
   itself evidence against the graph.
2. **graph_vs_native** — does the native drop agree with the drop circuit-tracer's *own*
   `feature_intervention` predicts on the replacement model for the same ablation? `invert`
   means the graph predicted a large effect the real model does not show; `agree_small` means
   both are small (redundancy); `agree_large` means both load-bearing.

Group interventions (jointly ablate the top-k set, or steer it to −2× activation, as the
attribution-graphs paper does) are compared the same way. This notebook loads the committed,
scrubbed bundles and re-derives every label **in-process, offline** — no GPU, no model, no
`circuit_tracer`, no torch. Install with `pip install -e .`.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'experiments').exists() else ROOT.parent
BUNDLES = ROOT / 'experiments' / 'circuit_tracer' / 'bundles'
prompts = sorted(p.name for p in BUNDLES.iterdir() if (p / 'edge_verdicts.json').exists())
print('committed verdict tables:', prompts)

In [ ]:
def load_table(name):
    return json.loads((BUNDLES / name / 'edge_verdicts.json').read_text())

def fmt(x, w=8, p=4):
    return (f'%{w}.{p}f' % x) if isinstance(x, (int, float)) else f'{str(x):>{w}}'

def show(name):
    t = load_table(name)
    s = t['summary']
    print(f"=== {name} ===  prompt {t['prompt']!r} -> target {t['target_text']!r}")
    print(f"  native-necessity: necessary {s['n_native_necessary']}  effect-absent "
          f"{s['n_native_effect_absent']}  inconclusive {s['n_inconclusive']}  (of {s['n_covered']})")
    print(f"  graph agreement : agree {s['n_agree_with_graph']}  invert {s['n_invert_vs_graph']}  "
          f"mixed {s['n_mixed_vs_graph']}  (of {s['n_with_graph_prediction']} with a prediction)")
    print(f"  error-node influence share: {s['error_node_influence_share']:.3f}")
    print(f"  {'layer':>5} {'pos':>4} {'feat':>6} {'native_d':>9} {'graph_d':>9} "
          f"{'graph_vs_native':>16} {'native_label':>20}")
    for r in t['rows']:
        if r['node_kind'] != 'feature':
            continue
        gp = r.get('graph_predicted_delta')
        print(f"  {r['layer']!s:>5} {r['position']!s:>4} {r['feature']!s:>6} "
              f"{fmt(r['native_logit_delta'],9)} {fmt(gp,9) if gp is not None else ' NA':>9} "
              f"{str(r.get('graph_vs_native')):>16} {r['verdict']:>20}")
    for g in t.get('group_interventions', []):
        print(f"  GROUP {g['name']:>14} (m={g['multiplier']:>4}): native {fmt(g['native_logit_delta'])} "
              f"graph {fmt(g['graph_predicted_delta'])} -> {g['graph_vs_native']} / {g['verdict']} "
              f"(top-token flip: {g['top_token_flip']})")
    err = [r for r in t['rows'] if r['node_kind'] == 'error'][0]
    print(f"  error nodes (uncovered): {err['measurements']['n_error_nodes']} nodes, "
          f"{err['measurements']['error_node_influence_share']:.3f} of node influence")

for name in prompts:
    show(name)
    print()

## Fresh-process re-derivation (offline, stdlib)

`verify_edge_bundle` re-checks each bundle's SHA-256 and re-derives every covered edge label
**and** its `graph_vs_native` tag, plus every group verdict, from the sealed scalars under the
frozen `DecisionRule` — no torch, no model, no `circuit_tracer`.

In [ ]:
from saturn_pub.interop.circuit_tracer import verify_edge_bundle

for name in prompts:
    r = verify_edge_bundle(BUNDLES / name)
    print(f"{name}: ok={r['ok']}  hash_ok={r['hash_ok']}  fingerprint_ok={r['fingerprint_ok']}  "
          f"verdict_ok={r['verdict_ok']}  group_ok={r['group_ok']}")
    assert r['ok'], r

## Reading

`native_effect_absent` on a single feature is only the first axis: under a redundant circuit
many features each carry little alone, so this is expected and not a graph failure. The
**`graph_vs_native`** column is the fair test — `invert` is where circuit-tracer's own
intervention predicts a large target-logprob drop on the replacement model that the real model
does not show (including the joint group interventions). `error-node influence share` is the
influence the transcoder features never expose. See [`docs/circuit-tracer.md`](../docs/circuit-tracer.md)
and [`experiments/circuit_tracer/README.md`](../experiments/circuit_tracer/README.md).